# 채보 모델 파인튜닝 + 페달 하이햇 + 크래시·라이드 — GPU 서버 실행 (S15P31A307-164, 작업 22·28·31)

ADTOF Frame_RNN을 E-GMD train 어쿠스틱·STAR training의 정답 드럼 트랙으로 더 학습시킨다. 학습 셋을 GPU 6번에서 **하나씩 차례로** 돌린다(10/30 셋을 동시에 돌렸더니 Jupyter가 버벅여서 설계 v0.4에서 바꿈).

- `tom_5cls`: 5클래스 그대로(설정 `configs/train/adtof_tom_5cls.yaml`). 탐 검출을 올리는 학습
- `tom_6cls`: 5클래스 + 페달 하이햇 출력(설정 `configs/train/adtof_tom_6cls.yaml`). 탐과 페달을 같이 올리는 학습
- `tom_7cls`: 6클래스에서 심벌을 크래시·라이드로 나눈 7출력(설정 `configs/train/adtof_tom_7cls.yaml`). 탐·페달·크래시·라이드를 같이 올리는 학습

학습과 별개로, 학습 전 가중치 특징에 크래시·라이드 규칙 후보(판정 구간 길이·크래시 가중치)를 바로 채점해 둔다.

학습이 도는 동안 drum2midi 학습 페달 모델을 MDB·GMD test에 돌려 페달 비교 대상을 만든다. **위에서부터 차례로 실행한다.** 셀마다 명령만 부르고 로직은 없다.

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에 올린다. 이 ipynb 파일은 `ssaplash/notebooks/`에 올린다. 서버에는 `.venv`·`.venv-drum2midi`·`ext/drum2midi`·데이터 링크(`data/e-gmd`, `data/star_drums/full`, `data/enst`, `data/gmd`, `data/mdb_drums`)·`data/models`·`data/rule_features/`(다섯 평가 데이터)가 이미 있다. 같은 GPU 6번을 다른 세션(-166 분리 평가 등)이 쓰고 있으면 끝날 때까지 기다린다.

**순서.** 1 설정 → 1-1 지금 상태(무엇을 치든 먼저, 여러 번) → 2 코드 덮어 풀기(1분) → 2-1 E-GMD 정답 시각 진단(1~2분, 어긋나 있으면 멈춤) → 3 학습·검증 데이터 점검(5~10분) → 4 본 학습을 한 번도 안 돌린 학습의 짧은 시험(각 2~3분) → 5 본 학습 대기열 시작(백그라운드, 하나씩, 끊긴 학습은 이어 감) → 6 진행 확인(여러 번) → 6-1 drum2midi 페달 모델 실행(백그라운드, **한 번만**) → 6-2 진행 확인 → 6-3 크래시·라이드 규칙 후보 채점(CPU, 학습과 같이) → 7 학습 전 가중치와 5가지·페달·크래시·라이드 F1 비교(30~60분) → 8 규칙 포함 점수용 특징 다시 저장(백그라운드, 대기열에서 끝남 표시가 난 학습부터, 여러 번 쳐도 됨) → 9 진행 확인 → 10 rule-eval(CPU 20~40분) → 10-1 페달 비교표 → 10-2 크래시·라이드 비교표 → 11 결과 묶기.

**끊기면.** 5·6-1·8번은 백그라운드라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번 → 1-1번을 치고 이어 간다. 학습이 도는지·끝났는지는 로그 파일이 있는지가 아니라 1-1·6번의 대기열 상태(pid, 끝남·실패 표시)로 본다. 학습이 '멈춤'이면 5번을 다시 치면 `last.pth`에서 이어 간다. 학습은 실행 폴더 `~/ssaplash/runs/164-train-1030`의 코드 사본으로 돌기 때문에, 학습 중에 2번 셀로 코드를 다시 풀어도 도는 학습은 바뀌지 않는다. 6-1번은 다시 치면 assert에 걸려 멈추고, 8번은 이미 시작한 것을 건너뛴다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import getpass
import os
import sys

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"   # nohup 로그가 바로바로 써지게
os.environ["MPLBACKEND"] = "Agg"
# 데이터 읽기 작업 하나가 코어 1개만 쓰게 함. 안 하면 numpy 행렬 곱이 작업마다 서버 코어 256개를 다 쓰려고 해서 서로 뺏음
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
VENV_PY = f"{AI}/.venv/bin/python"
ME = getpass.getuser()
# 학습 실행 폴더. 코드 사본과 대기열 상태 파일(status/)이 여기 생김. 코드를 바꿔 새로 돌릴 때만 이름을 바꿈
RUN_DIR = f"{WORK}/runs/164-train-1030"
QUEUE = f"{VENV_PY} {AI}/scripts/train_queue.py"
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
FFMPEG = f"{WORK}/tools/ffmpeg-7.0.2-amd64-static"
# 학습 이름 → (학습 설정, 규칙 포함 평가용 어댑터 설정). 설정을 바꿔 다시 돌리면 이름을 tom_5cls_w1 처럼 바꾸고 어댑터 yaml의 checkpoint_path도 맞춤
RUNS = {
    "tom_5cls": ("configs/train/adtof_tom_5cls.yaml", "configs/bench/transcribers/adtof_drumsep/adtof_drumsep_ft.yaml"),
    "tom_6cls": ("configs/train/adtof_tom_6cls.yaml", "configs/bench/transcribers/adtof_drumsep/adtof_drumsep_ft6.yaml"),
    "tom_7cls": ("configs/train/adtof_tom_7cls.yaml", "configs/bench/transcribers/adtof_drumsep/adtof_drumsep_ft7.yaml"),
}
OUT = {r: f"data/models/adtof_ft/{r}" for r in RUNS}
# 10/30 12:38 셋 동시 실행 때 로그(에폭 0~8). 이어 학습 로그는 RUN_DIR/status/<이름>.log
TRAIN_LOG = {r: f"{AI}/train_{r}.log" for r in RUNS}
FT_LOG = {r: f"{AI}/rule_features_ft_{r}.log" for r in RUNS}
D2M_LOG = f"{AI}/d2m_learned.log"
EVAL_DS = ("configs/bench/datasets/gmd_test.yaml configs/bench/datasets/mdb_drums.yaml "
           "configs/bench/datasets/enst_minus_one.yaml configs/bench/datasets/egmd_test_acoustic.yaml "
           "configs/bench/datasets/star_drums_test.yaml")
NPROC = os.cpu_count()
NUM_WORKERS = 8 if NPROC >= 128 else 4   # 학습을 하나씩 돌리므로 코어 8개 (설계 v0.4 4-3절 8). 그래도 버벅이면 4
os.makedirs(RESULTS, exist_ok=True)
%cd $AI
print("CPU 코어", NPROC, "→ 학습마다 데이터 읽기 작업", NUM_WORKERS)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!df -h $HOME | tail -1
!ls -la $UPLOAD/S15P31A307_code.zip
# 확인: 'CPU 코어 숫자 → …', GPU 한 줄(H200). memory.used가 수십 GB면 다른 작업이 도는 중이니 끝날 때까지 기다림. 코드 zip이 오늘 올린 것

In [ ]:
# 1-1. 지금 상태 (여러 번 쳐도 됨). 다른 셀을 치기 전에 먼저 침
!$QUEUE status --run-dir $RUN_DIR
!ps -u $ME -o pid,etime,pcpu,args | grep -E "[a]dtof-finetune|[t]rain_queue|[r]ule-features|[r]ule-eval|[d]rum2midi" | cut -c1-150
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
# 확인: 대기열을 띄우기 전이면 '대기열을 시작한 적 없음: …/runs/164-train-1030/status'.
#   띄운 뒤에는 '[대기열] 도는 중 pid …', '[서버] load … / 코어 256, 남은 메모리 …GB / …GB, 우리 데이터 읽기 작업 8개',
#   학습마다 '== tom_5cls: 도는 중 pid …' / '대기' / '끝남' / '실패 (종료 코드 …)' / '멈춤 (…)'.
#   ps 줄은 우리 계정에서 도는 학습·특징 저장·drum2midi. 대기열이 돌면 train_queue 1줄 + adtof-finetune 1줄.
#   GPU 사용률이 90%를 넘어도 다른 사용자 학습과 시간을 나눠 쓰는 것이라 그대로 진행. memory.used가 130,000MiB를 넘으면 기다림

In [ ]:
# 2. 코드 덮어 풀기. 기존 폴더를 지우지 않고 위에 덮어쓴다 (.venv·데이터 링크·특징 파일이 그대로 남음)
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!ls configs/train/adtof_tom_7cls.yaml configs/bench/transcribers/adtof_drumsep/adtof_drumsep_ft7.yaml docs/chart-rules/tom-newkits-2026-10-29/candidates_cymbal.yaml scripts/score_pedal.py scripts/score_cymbal.py
!.venv/bin/python -m ssaplash_ai.train.cli --help | head -5
!.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!grep -n "44:\|pedal" configs/bench/transcribers/external/drum2midi_learned.yaml
# 확인: 파일 5개가 그대로 찍히고, ssaplash-train 도움말에 adtof-finetune·adtof-eval, 'cuda True',
#       drum2midi_learned.yaml에 'args: ["--pedal", "auto"]'와 '44: HIHAT_PEDAL'.
#       ModuleNotFoundError면 .venv가 이 코드를 못 보는 것이니 `.venv/bin/pip install -e . --no-deps`

In [ ]:
# 2-1. E-GMD 정답 시각 진단 (CPU, 1~2분). 학습 전에 한 번만
#      새 킷 검증 때 저장한 특징(data/rule_features)의 예측 시각을 -100~+100ms 옮겨 가며 5가지 F1을 다시 잼.
#      E-GMD test 킥 0.71·스네어 0.50·하이햇 0.73이 같은 연주의 GMD test(0.9 이상)보다 낮은 원인이 정답 시각 어긋남인지 가르는 용도
!.venv/bin/python scripts/check_hit_offset.py --dataset configs/bench/datasets/egmd_test_acoustic.yaml configs/bench/datasets/gmd_test.yaml configs/bench/datasets/star_drums_test.yaml --n-songs 30
# 확인: 데이터셋마다 '== 이름: 곡 30개', 20ms 간격 f1 줄, '최고 이동량 +0ms (f1 …)'.
#       세 데이터셋 다 최고 이동량이 0ms(또는 ±10ms에서 f1 차이 0.01 안)면 시각 문제 아님. 그대로 3번으로.
#       E-GMD만 다른 이동량(예: +30ms)에서 f1이 0.05 넘게 오르면 E-GMD 정답 시각이 어긋난 것. 출력 전문을 세션에 붙여 넣고 멈춤
#       (로더 시각 보정 뒤 코드 zip을 다시 받아야 함. 그 전에 5번 본 학습을 시작하지 않음).
#       '특징 파일을 못 찾음'이면 `ls data/rule_features` 결과를 적어 둠

In [ ]:
# 3. 학습·검증 데이터 점검 (여러 번 쳐도 됨). E-GMD train은 21,294곡이라 5~10분
!ls -la data/e-gmd data/star_drums/full data/gmd data/enst 2>&1 | head -8
!.venv/bin/ssaplash-bench check-data configs/bench/datasets/egmd_validation_acoustic.yaml configs/bench/datasets/star_drums_validation.yaml configs/bench/datasets/star_drums_training_ismir04.yaml configs/bench/datasets/star_drums_training_mtg.yaml
!.venv/bin/ssaplash-bench check-data configs/bench/datasets/egmd_train_acoustic.yaml
# 확인: egmd_validation 3042곡, star_validation 119곡, star_training_ismir04 1228곡, star_training_mtg 4807곡,
#       egmd_train 21294곡. 전부 누락 0. 누락이 있으면 그 줄을 적어 두고 멈춤


In [ ]:
# 4. 본 학습을 한 번도 안 돌린 학습의 짧은 시험 (각 20스텝 × 1에폭). 데이터 읽기·학습·검증·문턱값 맞추기가 끝까지 도는지만 본다
for r, (cfg, _) in RUNS.items():
    if os.path.isfile(f"{OUT[r]}/last.pth"):
        print(r, "본 학습 가중치(last.pth)가 이미 있어 시험 건너뜀")
        continue
    smoke = f"data/models/adtof_ft/smoke_{r}"
    !rm -rf $smoke
    !.venv/bin/python -m ssaplash_ai.train.cli adtof-finetune --config $cfg --out $smoke --max-epochs 1 --steps-per-epoch 20 --num-workers $NUM_WORKERS
    !cut -d, -f1-8 $smoke/log.csv
# 확인 (학습마다):
#   '[train] 학습 곡 27329개(해시 …), 검증 곡 40개, 클래스 five(5)' (6cls는 'five_pedal(6)', 7cls는 'five_pedal_cym(7)'), '장치 cuda'
#   '[train] 기준값(기본 가중치 5클래스): kick 0.9xx, snare …, 페달 0.000 (…)'  ← 이 줄을 적어 둠. 학습 전 점수
#   '[train] 에폭 -1 …', '[train] 에폭 0 …', '[train] 문턱값 맞춤 […]', '[train] 끝.'
#   log.csv는 머리줄 + 에폭 -1 + 에폭 0, 세 줄.
#   6cls·7cls 에폭 -1·0 줄 끝의 '(기준값보다 0.02 넘게 떨어진 클래스 있음, 선택 안 함)'과 '경고: 기준값 조건을 넘는 에폭이 없어'는 시험에서는 정상
#   (페달은 하이햇, 크래시·라이드는 심벌 출력을 복사해 시작해서 처음엔 같은 소리를 두 번 냄). 7cls 에폭 줄에는 '크래시 … 라이드 …'도 붙음.
#   본 학습을 한 번이라도 돌린 학습은 '… 시험 건너뜀'(10/30 이후 셋 다). 오류가 나면 전문을 적어 두고 멈춤

In [ ]:
# 5. 본 학습 대기열 시작 (백그라운드). tom_5cls → tom_6cls → tom_7cls 순서로 하나씩, nice 19, 데이터 읽기 작업 NUM_WORKERS개
#    RUN_DIR에 코드 사본을 풀어 그 사본으로 학습함. 끊긴 학습은 data/models/adtof_ft/<이름>/last.pth에서 이어 감(--resume)
#    대기열이 이미 돌고 있으면 새로 띄우지 않으므로 다시 쳐도 됨. 끝남 표시가 난 학습은 건너뜀
runs = " ".join(f"--run {r}={cfg}" for r, (cfg, _) in RUNS.items())
!$QUEUE start --run-dir $RUN_DIR --zip $UPLOAD/S15P31A307_code.zip --shared-ai $AI --python $VENV_PY --num-workers $NUM_WORKERS $runs
# 확인 (처음): '코드 사본 풂: /home/j-k15a307/ssaplash/runs/164-train-1030/S15P31A307/AI (파일 6xx개)',
#              'data 링크: …/runs/164-train-1030/S15P31A307/AI/data → /home/j-k15a307/ssaplash/S15P31A307/AI/data',
#              '대기열 시작 pid 숫자: tom_5cls → tom_6cls → tom_7cls, 데이터 읽기 작업 8개, nice 19'
#   대기열이 이미 돌면 '대기열이 이미 돌고 있음 pid …'. 그대로 6번으로
#   대기열이 꺼진 뒤 다시 치면 '코드 사본 그대로 씀: …'. 새 코드로 돌리려면 1번 셀 RUN_DIR 이름을 바꾸고 1번부터 다시 침
#   '학습 설정을 못 찾음'이면 코드 zip이 예전 것. 2번 셀 출력의 파일 목록을 확인

In [ ]:
# 6. 진행 확인 (여러 번 쳐도 됨). 학습 하나가 끝나면 대기열이 다음 학습을 띄움
!$QUEUE status --run-dir $RUN_DIR
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
# 확인: '[대기열] 도는 중 pid …', '[서버] load …, 우리 데이터 읽기 작업 8개'.
#   10/30 처음 이어 돌릴 때 tom_5cls 줄: '[train] 이어 학습: 에폭 9부터, tom_f1 최고 0.716@7, 안 오른 에폭 1, 학습률 2.00e-05. 옵티마이저 상태는 새로 시작'
#   (tom_6cls는 'tom_pedal_f1 최고 0.571@7, 안 오른 에폭 1, 학습률 1.00e-04', tom_7cls는 'focus_f1 최고 0.640@8, 안 오른 에폭 0, 학습률 1.00e-04')
#   그 뒤 '[train] 에폭 N 손실 … 검증 f1 … 탐 …' 줄과 'log.csv: N,…' 줄이 늘어남. 차례가 안 온 학습은 '대기'.
#   끝난 학습은 '끝남'이고 로그에 '[train] 문턱값 맞춤 …'. 그 학습부터 8번 셀(규칙 포함 특징 저장)을 쳐도 됨
#   '실패 (종료 코드 …)'면 RUN_DIR/status/<이름>.log 끝을 적어 둠. 대기열은 다음 학습으로 넘어감
#   '멈춤'은 끝 표시 없이 꺼진 것. 5번 셀을 다시 치면 그 자리에서 이어 감
#   Jupyter가 다시 버벅이면 아래 줄 주석을 풀어 대기열을 끄고(이어 학습 가능), 1번 셀 NUM_WORKERS를 4로 바꿔 1번 → 5번을 다시 침
# !$QUEUE stop --run-dir $RUN_DIR

In [ ]:
# 6-1. drum2midi 학습 페달 모델 실행 (백그라운드, 한 번만, 학습과 같이 돌려도 됨). MDB 23곡 → GMD test 45개 순서
import subprocess

assert not os.path.exists(D2M_LOG), "이미 시작했음. 6-2번 셀로 진행을 확인한다"
assert os.path.isdir(".venv-drum2midi") and os.path.isdir("ext/drum2midi"), "drum2midi 환경이 없음"
env = dict(os.environ, PATH=f"{FFMPEG}:{os.environ['PATH']}")
script = " && ".join(
    f".venv/bin/ssaplash-bench run-transcription --models configs/bench/transcribers/external/drum2midi_learned.yaml "
    f"--dataset configs/bench/datasets/{ds}.yaml --input drums --run-id d2m-learned-{tag} --out data/results --mlflow-uri none"
    for ds, tag in (("mdb_drums", "mdb"), ("gmd_test", "gmd"))
)
proc = subprocess.Popen(["nohup", "bash", "-c", script], cwd=AI, stdout=open(D2M_LOG, "w"),
                        stderr=subprocess.STDOUT, env=env, start_new_session=True)
print("시작. pid", proc.pid, "로그", D2M_LOG)
# 확인: '시작. pid 숫자'. 지난번(규칙 페달) 실행은 두 데이터 합쳐 1시간 안쪽이었음

In [ ]:
# 6-2. drum2midi 진행 확인 (여러 번). results.csv 줄 수가 MDB 24·GMD 46(머리줄 포함)이 되면 끝
!wc -l data/results/d2m-learned-mdb/results.csv data/results/d2m-learned-gmd/results.csv 2>/dev/null
!tail -3 $D2M_LOG
!grep -c "error" data/results/d2m-learned-*/results.csv 2>/dev/null || true
!pgrep -f "[d]2m-learned" > /dev/null && echo "아직 도는 중" || echo "끝났거나 안 돌고 있음"
# 확인: 줄 수가 늘어남. 로그에 'ffmpeg' 오류가 있으면 FFMPEG 경로를, 'ModuleNotFoundError'면 .venv-drum2midi를 적어 둠

In [ ]:
# 6-3. 크래시·라이드 규칙 후보 채점 (CPU, 학습과 같이 돌려도 됨, 30분~1시간). 학습 전 가중치로 저장한 특징(data/rule_features)에 심벌 규칙만 바꿔 채점
!.venv/bin/ssaplash-bench rule-eval --features data/rule_features --dataset $EVAL_DS --candidates docs/chart-rules/tom-newkits-2026-10-29/candidates_cymbal.yaml --out data/results/rule-eval-base-cymbal --two-tom-upper MID_TOM --mlflow-uri none
!cut -d, -f1,2,9,10 data/results/rule-eval-base-cymbal/candidates.csv
# 확인: '[rule-eval] 결과 → data/results/rule-eval-base-cymbal/summary.md'. 아래 표는 후보 8개(pk_gap30·cw40·cw150·cw300·cx05·cx07·cx14·cx20) × 데이터셋 5개의
#       f1CRASH·f1RIDE. pk_gap30 줄이 10/30 당시 규칙(그 뒤 크래시 가중치 0.7을 더함). 오류가 나면 마지막 20줄을 적어 둠

In [ ]:
# 7. 학습 전 가중치 vs 파인튜닝 가중치 셋, 5가지·페달·크래시·라이드 F1 (GMD test · MDB · ENST · E-GMD test · STAR test). 30~60분
ck = " ".join(f"{OUT[r]}/best.pth" for r in RUNS)
!.venv/bin/python -m ssaplash_ai.train.cli adtof-eval --checkpoint $ck --dataset $EVAL_DS --out data/results/adtof-eval-ft
!.venv/bin/python -m ssaplash_ai.train.cli adtof-eval --dataset $EVAL_DS --out data/results/adtof-eval-base
!cat data/results/adtof-eval-ft/summary.md
!cat data/results/adtof-eval-base/summary.md
# 확인: 위 표는 데이터셋 5개 × 체크포인트 3개(파일 이름이 다 best.pth라 순서로 구분: 각 데이터셋에서 tom_5cls, tom_6cls, tom_7cls 순서), 아래는 5개 × 기본 가중치.
#       7cls 줄에만 f1_crash·f1_ride·focus_f1 칸이 차 있음.
#       '[adtof-eval] … 페달 0.xxx (정답 페달을 HIHAT으로 N개)' 줄도 같이 찍힘.
# 보는 것: tom_f1이 MDB·ENST에서 올랐는지, f1_kick·f1_snare·f1_hihat·f1_cymbal이 어느 데이터에서도 0.02 넘게 안 떨어졌는지,
#          tom_6cls·tom_7cls의 pedal_f1이 MDB·GMD에서 얼마인지, tom_7cls의 f1_crash·f1_ride

In [ ]:
# 8. 규칙 포함 점수용 특징 다시 저장 (백그라운드, nice 19). 대기열에서 끝남 표시가 난 학습만 시작하고, 이미 시작한 것·아직 학습 중인 것은 건너뜀. 여러 번 쳐도 됨
#    분리 트랙은 data/rule_features에서 복사하고 채보 모델만 다시 돌림
import subprocess

for r, (_, model_yaml) in RUNS.items():
    if os.path.exists(FT_LOG[r]):
        print(r, "이미 시작했음, 건너뜀")
        continue
    done = os.path.isfile(f"{RUN_DIR}/status/{r}.DONE")
    if not (done and os.path.isfile(f"{OUT[r]}/best.pth")):
        print(r, "학습이 아직 안 끝남, 건너뜀 (6번 셀로 확인 뒤 다시 침)")
        continue
    cmd = ["nice", "-n", "19", ".venv/bin/ssaplash-bench", "rule-features", "--model", model_yaml,
           "--out", f"data/rule_features_ft_{r}", "--stems-from", "data/rule_features",
           "--dataset", *EVAL_DS.split()]
    proc = subprocess.Popen(
        ["nohup", *cmd], cwd=AI, stdout=open(FT_LOG[r], "w"), stderr=subprocess.STDOUT,
        env=dict(os.environ), start_new_session=True,
    )
    print("시작.", r, "pid", proc.pid, "로그", FT_LOG[r])
# 확인: 끝난 학습마다 '시작. tom_… pid …'. 안 끝난 것은 '… 건너뜀'이라 나중에 이 셀을 다시 침. 어댑터 yaml의 checkpoint_path가 data/models/adtof_ft/<이름>/best.pth라 이름을 바꿨으면 yaml도 맞춤

In [ ]:
# 9. 진행 확인 (여러 번). 학습마다 곡 수가 45 / 23 / 64 / 1170 / 31이 되면 끝
import glob

for r in RUNS:
    print("=====", r)
    if not os.path.exists(FT_LOG[r]):
        print("아직 시작 안 함 (학습이 끝나면 8번 셀)")
        continue
    for d in ("gmd_test", "mdb_drums", "enst_minus_one", "egmd_test_acoustic", "star_drums_test"):
        # ENST 곡 id(drummer1/session1/28)처럼 곡이 하위 폴더에 있어서 아래까지 셈
        print(f"{d}: {len(glob.glob(f'data/rule_features_ft_{r}/{d}/**/*.npz', recursive=True))}개")
    !tail -2 {FT_LOG[r]}
    !grep -c "분리 트랙 재사용" {FT_LOG[r]} || true
    !grep -c '실패 [1-9]' {FT_LOG[r]} || true
!pgrep -fc "[s]saplash-bench rule-features" || true
# 확인: 개수가 늘고 '분리 트랙 재사용' 수가 저장한 곡 수와 같음(0이면 분리를 새로 돌리는 중이라 몇 시간 걸림). 실패 줄은 0. 마지막 숫자는 도는 작업 수

In [ ]:
# 10. 규칙 비교 띄우기 (CPU, 백그라운드, nice 19). 저장한 특징만 읽어 채점해서 GPU는 안 씀
#     작업 7개: 후보 5개 채점 4개(학습 전 가중치·학습 셋) + 탐 기준값 40개 훑기 3개. 서로 안 기다리고 동시에 돎
#     작업 하나가 다섯 데이터의 특징을 전부 메모리에 올려서(약 100GB) 남은 메모리로 동시 개수를 정함.
#     메모리가 모자라 못 띄운 것은 아래 '10 확인' 셀이 앞 작업이 특징을 다 읽은 뒤 띄움. 작업마다 코어 1개라 Jupyter에 영향 적음
import glob
import subprocess

import yaml

CAND = "docs/chart-rules/tom-newkits-2026-10-29"
DS_NAMES = ("gmd_test", "mdb_drums", "enst_minus_one", "egmd_test_acoustic", "star_drums_test")
RULE_JOBS = {"base-pedal": ("data/rule_features", "candidates.yaml")}
RULE_JOBS.update({f"ft-{r}": (f"data/rule_features_ft_{r}", "candidates.yaml") for r in RUNS})
RULE_JOBS.update({f"ft-{r}-threshold": (f"data/rule_features_ft_{r}", "candidates_threshold.yaml") for r in RUNS})
RULE_LOG = {n: f"{AI}/rule_eval_{n}.log" for n in RULE_JOBS}


def pid_alive(pid):
    # 끝났는데 회수 안 된 좀비 프로세스는 죽은 것으로
    try:
        with open(f"/proc/{pid}/stat") as f:
            return f.read().rsplit(")", 1)[1].split()[0] != "Z"
    except (OSError, IndexError):
        return False


def rule_log_text(n):
    return open(RULE_LOG[n], encoding="utf-8", errors="replace").read()


def rule_state(n):
    if not os.path.exists(RULE_LOG[n]):
        return "대기"
    if "[rule-eval] 결과 →" in rule_log_text(n):
        return "끝남"
    try:
        pid = int(open(RULE_LOG[n] + ".pid").read())
    except (OSError, ValueError):
        return "실패"
    return "도는 중" if pid_alive(pid) else "실패"


def rule_loading(n):
    # 특징을 아직 읽는 중이면 메모리 사용량이 더 늘어남
    return rule_log_text(n).count("특징 읽음") < len(DS_NAMES)


def job_mem_gb(feats):
    # 스템이 파일에는 float16, 메모리에는 float32라 파일 크기의 약 2.2배
    total = sum(os.path.getsize(p) for d in DS_NAMES for p in glob.glob(f"{feats}/{d}/**/*.npz", recursive=True))
    return total * 2.2 / 1e9


def mem_avail_gb():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) / 1e6
    return 0.0


def launch_rule_jobs():
    avail = mem_avail_gb() * 0.7   # 30%는 다른 사용자·Jupyter 몫으로 남김
    for n, (feats, cand) in RULE_JOBS.items():
        if rule_state(n) != "대기":
            continue
        need = job_mem_gb(feats)
        if need > avail:
            print(f"{n}: 메모리가 모자라 대기 (필요 약 {need:.0f}GB, 쓸 수 있는 것 {avail:.0f}GB)")
            continue
        cmd = ["nice", "-n", "19", ".venv/bin/ssaplash-bench", "rule-eval", "--features", feats,
               "--dataset", *EVAL_DS.split(), "--candidates", f"{CAND}/{cand}",
               "--out", f"data/results/rule-eval-{n}", "--two-tom-upper", "MID_TOM", "--mlflow-uri", "none"]
        proc = subprocess.Popen(
            ["nohup", *cmd], cwd=AI, stdout=open(RULE_LOG[n], "w"), stderr=subprocess.STDOUT,
            env=dict(os.environ), start_new_session=True,
        )
        with open(RULE_LOG[n] + ".pid", "w") as f:
            f.write(str(proc.pid))
        avail -= need
        print(f"시작. {n} pid {proc.pid} (메모리 약 {need:.0f}GB)")


print(f"[서버] 남은 메모리 {mem_avail_gb():.0f}GB")
launch_rule_jobs()
# 확인: '시작. base-pedal pid …'처럼 줄이 나옴. 메모리가 넉넉하면 7줄, 아니면 일부는 '메모리가 모자라 대기'. 세 학습의 9번 곡 수가 다 찬 뒤에 친다.
#       다시 쳐도 이미 돌거나 끝난 작업은 안 띄움. 실패한 작업을 다시 돌리려면 그 로그(AI/rule_eval_<이름>.log)를 지우고 침

In [ ]:
# 10 확인. 규칙 비교 진행 (몇 번이고 쳐도 됨). 메모리가 모자라 대기한 작업은 도는 작업이 특징을 다 읽은 뒤 여기서 띄움
n_cand = {c: len(yaml.safe_load(open(f"{CAND}/{c}", encoding="utf-8"))["candidates"]) for c in ("candidates.yaml", "candidates_threshold.yaml")}
for n, (feats, cand) in RULE_JOBS.items():
    st = rule_state(n)
    if st == "대기":
        print(f"{n:22s} 대기")
        continue
    text = rule_log_text(n)
    done = text.count("f1Tom=")
    read = text.count("특징 읽음")
    print(f"{n:22s} {st:5s} 특징 읽은 데이터 {read}/{len(DS_NAMES)}, 채점 {done}/{n_cand[cand] * len(DS_NAMES)}")
    if st == "실패":
        print("    로그 끝:", *text.strip().splitlines()[-3:], sep="\n    ")
if not any(rule_state(n) == "도는 중" and rule_loading(n) for n in RULE_JOBS):
    launch_rule_jobs()
print(f"[서버] 남은 메모리 {mem_avail_gb():.0f}GB, 도는 rule-eval", sum(rule_state(n) == "도는 중" for n in RULE_JOBS), "개")
if all(rule_state(n) == "끝남" for n in RULE_JOBS):
    for n in ["base-pedal", *[f"ft-{r}" for r in RUNS]]:
        print("==", n)
        !grep pk_gap30 data/results/rule-eval-{n}/candidates.csv | cut -d, -f1-12
# 확인: 작업마다 한 줄. 7개가 다 '끝남'이면 맨 아래 pk_gap30 네 묶음(데이터 5줄씩)이 나오고, 그다음 10-1·10-2. 탐 3종 F1은 f1Tom 열.
#       '실패'가 나오면 로그 끝 3줄을 붙여 주면 됨. 특징 읽는 데 작업마다 수십 분, 채점은 기준값 훑기 3개가 제일 오래 걸림

In [ ]:
# 10-1. 페달 비교표 (CPU, 몇 초). 학습 전 가중치·tom_5cls·tom_6cls(규칙 pk_gap30)와 drum2midi 학습 페달 모델
!.venv/bin/python scripts/score_pedal.py --rule-eval base=data/results/rule-eval-base-pedal@pk_gap30 --rule-eval tom_5cls=data/results/rule-eval-ft-tom_5cls@pk_gap30 --rule-eval tom_6cls=data/results/rule-eval-ft-tom_6cls@pk_gap30 --rule-eval tom_7cls=data/results/rule-eval-ft-tom_7cls@pk_gap30 --run d2m_learned=data/results/d2m-learned-mdb --run d2m_learned=data/results/d2m-learned-gmd --baseline base --out data/results/pedal-compare.md
# 확인: 데이터셋마다 표 하나(방식 5개 행. drum2midi는 MDB·GMD에만), 끝에 '## 채택 판정' 아래 방식마다 '채택 가능' 또는 '기준 못 넘음: …'.
#       base·tom_5cls는 페달을 안 내므로 페달 F1 0이 정상

In [ ]:
# 10-2. 크래시·라이드 비교표 (CPU, 몇 초). 10/30 당시 규칙·규칙 후보 7개·세 학습(규칙 pk_gap30, 7cls는 모델이 직접 나눔)
CY = "data/results/rule-eval-base-cymbal"
cands = " ".join(f"--rule-eval {c}={CY}@{c}" for c in ("cw40", "cw150", "cw300", "cx05", "cx07", "cx14", "cx20"))
fts = " ".join(f"--rule-eval {r}=data/results/rule-eval-ft-{r}@pk_gap30" for r in RUNS)
!.venv/bin/python scripts/score_cymbal.py --rule-eval now={CY}@pk_gap30 $cands $fts --baseline now --out data/results/cymbal-compare.md
# 확인: 데이터셋마다 표 하나(방식 11개 행: now, 규칙 후보 7개, tom_5cls·tom_6cls·tom_7cls), 끝에 '## 채택 판정' 아래 방식마다 '채택 가능' 또는 '기준 못 넘음: …'

In [ ]:
# 11. 결과 묶기. results 폴더의 finetune_results.zip 을 내려받아 로컬 AI/data/results/ 에 둔다 (푸는 것은 세션에서. zip 안은 폴더·파일 이름만)
!rm -f $RESULTS/finetune_results.zip
!rm -rf data/results/train-queue-status && cp -r $RUN_DIR/status data/results/train-queue-status
items = " ".join([*OUT.values(), "data/results/adtof-eval-ft", "data/results/adtof-eval-base",
                  "data/results/rule-eval-base-pedal", *[f"data/results/rule-eval-ft-{r}" for r in RUNS],
                  *[f"data/results/rule-eval-ft-{r}-threshold" for r in RUNS],
                  "data/results/d2m-learned-mdb", "data/results/d2m-learned-gmd", "data/results/pedal-compare.md",
                  "data/results/rule-eval-base-cymbal", "data/results/cymbal-compare.md", "data/results/train-queue-status",
                  *[os.path.basename(p) for p in (*TRAIN_LOG.values(), *FT_LOG.values(), D2M_LOG, *RULE_LOG.values())]])
!$PY -m zipfile -c $RESULTS/finetune_results.zip $items
!ls -la $RESULTS/finetune_results.zip
missing = [p for p in items.split() if not os.path.exists(p)]
print("zip에 빠진 것:", missing or "없음")
# 확인: 'zip에 빠진 것: 없음'. 빠진 것이 있으면 그 셀을 안 친 것이니 치고 다시 11. 수백 MB짜리 zip 하나 (best.pth·last.pth 각 2MB, drum2midi 결과의 입력 wav 포함)

In [ ]:
# 12. ENST drummer_1 정답 없는 탐 자리 확인 (CPU, nice 19, 5~15분). 셀 1 다음 바로 쳐도 됨
#     정답 탐이 없는데 남은 탐 예측마다 ±50ms 안 ENST 원래 주석(bd 킥·sd 스네어·chh 하이햇·sweep 브러시 쓸기 등)을 셈
#     scripts/check_false_toms.py는 코드 zip에 들어 있음
FALSE_OUT = "data/results/false-toms-enst-drummer_1.md"
feats = " ".join(["--features base=data/rule_features", *[f"--features {r}=data/rule_features_ft_{r}" for r in RUNS]])
!nice -n 19 .venv/bin/python scripts/check_false_toms.py $feats --dataset configs/bench/datasets/enst_minus_one.yaml --song-prefix drummer_1 --candidates docs/chart-rules/tom-newkits-2026-10-29/candidates.yaml --candidate pk_gap30 --two-tom-upper MID_TOM --out $FALSE_OUT
!cd data/results && $PY -m zipfile -c $RESULTS/false_toms.zip false-toms-enst-drummer_1.md && ls -la $RESULTS/false_toms.zip
# 확인: 곡마다 '[false-toms] drummer_1/…: base 숫자, tom_5cls 숫자, …' 21줄, 끝에 '결과 →' 한 줄과 zip 한 줄.
#       tom_6cls 숫자를 다 더하면 3,268이어야 rule-eval과 같음. results 폴더의 false_toms.zip 을 로컬 AI/data/results/ 에 둠
#       'No such file'이 나오면 스크립트를 안 올린 것. 다른 오류는 끝 몇 줄을 붙여 주면 됨

In [ ]:
# 13. 킥과 겹친 탐 지우기 후보 채점 (CPU, 백그라운드, nice 19). 셀 1 → 2(코드 덮어 풀기) → 10(함수 정의) 다음에 침
#     후보 7개(pk_gap30 기준 + 킥 겹침 지우기 6개) × 학습 전 가중치·학습 셋 = 작업 4개. 10번과 같이 남은 메모리로 동시 개수를 정함
#     10번 셀은 이미 끝난 작업을 다시 띄우지 않으니 함수 정의용으로 쳐도 됨
KD_CAND = "docs/chart-rules/tom-kick-drop-2026-10-31/candidates.yaml"
KD_JOBS = {"kd-base": "data/rule_features", **{f"kd-ft-{r}": f"data/rule_features_ft_{r}" for r in RUNS}}
RULE_LOG.update({n: f"{AI}/rule_eval_{n}.log" for n in KD_JOBS})


def launch_kd_jobs():
    avail = mem_avail_gb() * 0.7   # 30%는 다른 사용자·Jupyter 몫으로 남김
    for n, feats in KD_JOBS.items():
        if rule_state(n) != "대기":
            continue
        need = job_mem_gb(feats)
        if need > avail:
            print(f"{n}: 메모리가 모자라 대기 (필요 약 {need:.0f}GB, 쓸 수 있는 것 {avail:.0f}GB)")
            continue
        cmd = ["nice", "-n", "19", ".venv/bin/ssaplash-bench", "rule-eval", "--features", feats,
               "--dataset", *EVAL_DS.split(), "--candidates", KD_CAND,
               "--out", f"data/results/rule-eval-{n}", "--two-tom-upper", "MID_TOM", "--mlflow-uri", "none"]
        proc = subprocess.Popen(
            ["nohup", *cmd], cwd=AI, stdout=open(RULE_LOG[n], "w"), stderr=subprocess.STDOUT,
            env=dict(os.environ), start_new_session=True,
        )
        with open(RULE_LOG[n] + ".pid", "w") as f:
            f.write(str(proc.pid))
        avail -= need
        print(f"시작. {n} pid {proc.pid} (메모리 약 {need:.0f}GB)")


!grep -c tom_kick_drop_ms src/ssaplash_ai/bench/transcribers/adtof_drumsep_transcriber.py
print(f"[서버] 남은 메모리 {mem_avail_gb():.0f}GB")
launch_kd_jobs()
# 확인: 맨 위 숫자가 0이 아님(새 코드가 풀렸음. 0이면 2번 셀부터 다시). 그다음 '시작. kd-base pid …' 같은 줄,
#       메모리가 넉넉하면 4줄, 아니면 일부는 '메모리가 모자라 대기'. 다시 쳐도 이미 돌거나 끝난 작업은 안 띄움

In [ ]:
# 13 확인. 진행 (몇 번이고 쳐도 됨). 대기한 작업은 도는 작업이 특징을 다 읽은 뒤 여기서 띄움
import csv

if "KD_CAND" not in globals():
    raise SystemExit("바로 위 13번 셀(# 13. 킥과 겹친 탐 지우기 …)을 먼저 침")
n_kd = len(yaml.safe_load(open(KD_CAND, encoding="utf-8"))["candidates"])
for n in KD_JOBS:
    st = rule_state(n)
    if st == "대기":
        print(f"{n:16s} 대기")
        continue
    text = rule_log_text(n)
    print(f"{n:16s} {st:5s} 특징 읽은 데이터 {text.count('특징 읽음')}/{len(DS_NAMES)}, 채점 {text.count('f1Tom=')}/{n_kd * len(DS_NAMES)}")
    if st == "실패":
        print("    로그 끝:", *text.strip().splitlines()[-3:], sep="\n    ")
if not any(rule_state(n) == "도는 중" and rule_loading(n) for n in KD_JOBS):
    launch_kd_jobs()
print(f"[서버] 남은 메모리 {mem_avail_gb():.0f}GB, 도는 작업", sum(rule_state(n) == "도는 중" for n in KD_JOBS), "개")
if all(rule_state(n) == "끝남" for n in KD_JOBS):
    for n in KD_JOBS:
        print("==", n, "탐 3종 F1 (MDB · ENST · STAR · E-GMD · GMD), 킥 겹침으로 버린 탐 수")
        rows = {(x["candidate"], x["dataset"]): x for x in csv.DictReader(open(f"data/results/rule-eval-{n}/candidates.csv", encoding="utf-8"))}
        order = ("mdb_drums", "enst_minus_one", "star_drums_test", "egmd_test_acoustic", "gmd_test")
        for c in dict.fromkeys(k[0] for k in rows):
            print(f"  {c:11s}", " ".join(f"{float(rows[(c, d)]['f1Tom']):.3f}" for d in order),
                  " 버림", " ".join(f"{float(rows[(c, d)]['tomKickDropped']):.0f}" for d in order))
# 확인: 작업마다 한 줄. 4개가 다 '끝남'이면 작업마다 후보 7줄씩 표가 나옴. 그 표를 붙여 주고 13-1로 묶음.
#       '실패'가 나오면 로그 끝 3줄을 붙여 주면 됨. 작업 하나에 1~2시간

In [ ]:
# 13-1. 결과 묶기. results 폴더의 kick_drop_results.zip 을 로컬 AI/data/results/ 에 둔다 (zip 안은 폴더·파일 이름만)
!rm -f $RESULTS/kick_drop_results.zip
kd_items = " ".join([*[f"data/results/rule-eval-{n}" for n in KD_JOBS], *[os.path.basename(RULE_LOG[n]) for n in KD_JOBS]])
!$PY -m zipfile -c $RESULTS/kick_drop_results.zip $kd_items
!ls -la $RESULTS/kick_drop_results.zip
missing = [p for p in kd_items.split() if not os.path.exists(p)]
print("zip에 빠진 것:", missing or "없음")
# 확인: 'zip에 빠진 것: 없음'과 수 MB짜리 zip 한 줄